In [ ]:
# !mkdir src

- Since we are mainly interested in sensible autocompletion for the the data science libraries, it makes sense to give more weight to training samples that make more use of these libraries.
- We can easily identify these examples through the use of keywords such as `plt`, `pd`, `sk`, `fit`, and `predict`, which are the most frequent import names for `matplotlib.pyplot`, `pandas`, and `sklearn` as well as the fit/predict pattern of the latter.
- If these are each represented as a single token, we can easily check if they occur in the input sequence.
- Tokens can have a whitespace prefix, so we’ll also check for those versions in the tokenizer vocabulary.
- To verify that it works, we’ll add one test token which should be split into multiple tokens.

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
dataset_dict = dict(path = "huggingface-course/codeparrot-ds")
train_ds_size = 300
valid_ds_size = 100
context_length = 128
tokenizer_ckpt = "huggingface-course/code-search-net-tokenizer"
model_checkpoint = "gpt-2"
push_to_hub=False
train_bs = 8
eval_bs = 16
eval_on_start = True
gacc_steps = 8
num_train_epochs = 1
lr_scheduler_type="cosine"
lr = 5e-4
max_grad_norm = 1.0
task = "clm"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-xxsmall-pretrained-{task}-{dataset_dict['path'].split("/")[-1]}-accelerate"

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

In [ ]:
%%writefile src/data.py

from datasets import load_dataset, DatasetDict
from src.config import dataset_dict, train_ds_size, valid_ds_size, context_length

def load_raw_datasets():
    ds_train = load_dataset(f"{dataset_dict["path"]}-train", split="train")
    ds_valid = load_dataset(f"{dataset_dict["path"]}-valid", split="validation")
    
    # TODO: remove shuffle and use full size if resources allow
    raw_datasets = DatasetDict(
        {
            "train": ds_train.shuffle(seed=42).select(range(train_ds_size)),
            "valid": ds_valid.shuffle(seed=42).select(range(valid_ds_size))
        }
    )
    return raw_datasets

def create_tokenized_datasets(tokenizer, raw_datasets):
    def tokenize(batch, tokenizer):
        outputs = tokenizer(
            batch["content"],
            truncation=True,
            max_length=context_length,
            return_overflowing_tokens=True,
            return_length=True,
        )
        return {
            "input_ids": [
                ids for l, ids in zip(outputs["length"], outputs["input_ids"])
                if l == context_length
            ]
        }
    
    tokenized_datasets = raw_datasets.map(
        tokenize,
        batched=True,
        fn_kwargs = {"tokenizer": tokenizer},
        remove_columns=raw_datasets["train"].column_names
    )
    return tokenized_datasets

In [ ]:
%%writefile src/train.py

from transformers import GPT2LMHeadModel, AutoConfig
from src.config import context_length

def load_model(tokenizer):
    config = AutoConfig.from_pretrained(
        "gpt2",
        vocab_size=len(tokenizer),
        n_ctx=context_length,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    model = GPT2LMHeadModel(config)
    return model


def get_grouped_params(
    model,
    no_decay=["bias", "LayerNorm.weight"],
    weight_decay = 0.1
):
    # TODO: Why don't those layers need WD?
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd, "weight_decay": weight_decay},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

- We can now write a custom loss function that takes the input sequence, the logits, and the key tokens we just selected as inputs.
- First we need to align the logits and inputs
    - the input sequence shifted by one to the right forms the labels, since the next token is the label for the current token.
    - We can achieve this by starting the labels from the second token of the input sequence, since the model does not make a prediction for the first token anyway.
    - Then we cut off the last logit, as we don’t have a label for the token that follows the full input sequence.
- With that we can compute the loss per sample and count the occurrences of all keywords in each sample.
- Finally, we calculate the weighted average over all samples using the occurrences as weights.
    - Since we don’t want to throw away all the samples that have no keywords, we add 1 to the weights.

In [ ]:
%%writefile src/evaluation.py

from torch.nn import CrossEntropyLoss
import torch, math
from tqdm.auto import tqdm

def get_keytoken_ids(tokenizer):
    keytoken_ids = []
    for keyword in ["plt", "pd", "sk", "fit", "predict", " plt",
                    " pd", " sk", " fit", " predict", "testtest"]:
        ids = tokenizer([keyword]).input_ids[0]
        if len(ids) == 1:
            keytoken_ids.append(ids[0])
        else:
            print(f"Keyword has not single token: {keyword}")
    return keytoken_ids

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    # Shift so that tokens < n predict n
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()
    # TODO: why do we call contiguous()??
    # Calculate per-token loss
    loss_fct = CrossEntropyLoss(reduction='none')
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    # Resize and average loss per sample
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    # Calculate and scale weighting
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)
    # Calculate weighted average
    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss


def run_eval(model, eval_dataloader, accelerator):
    model.eval()
    print(f"[rank={accelerator.process_index}] eval_dataloader batches: {len(eval_dataloader)}")
    
    losses = []
    batch_sizes = []
    progress_bar = tqdm(
        eval_dataloader,
        desc="Eval Loop",
        disable=not accelerator.is_main_process,
        leave=False,
    )
    
    for batch in eval_dataloader:
        bs = batch["input_ids"].shape[0]
        with torch.no_grad():
            with accelerator.autocast():
                outputs = model(batch["input_ids"], labels=batch["input_ids"])
        
        # gather both loss and batch size across processes
        loss_t = accelerator.gather_for_metrics(outputs.loss.unsqueeze(0))   # shape [num_gpus]
        bs_t   = accelerator.gather_for_metrics(
            torch.tensor([bs], device=accelerator.device)
        )                                                                      # shape [num_gpus]
        losses.append(loss_t)
        batch_sizes.append(bs_t)
        progress_bar.update(1)

    all_losses     = torch.cat(losses)       # shape [num_batches * num_gpus]
    all_batch_sizes = torch.cat(batch_sizes) # shape [num_batches * num_gpus]

    # weighted average: sum(loss_i * bs_i) / sum(bs_i)
    eval_loss = (all_losses * all_batch_sizes).sum() / all_batch_sizes.sum()
    eval_loss = eval_loss.item()
    
    return { "eval_loss": eval_loss, "perplexity": math.exp(eval_loss)
    }

### What happens inside the loss function?

In [ ]:
# alpha=1.0
# from src.config import keytoken_ids
# sample_batch = next(iter(train_dataloader))
# for k,v in sample_batch.items():
#     print(f"{k} | {v.shape}")
# inputs = sample_batch["input_ids"]
# logits = model(inputs).logits
# print(f"logits.shape: {logits.shape}")
# shift_labels = inputs[..., 1:].contiguous()
# shift_logits = logits[..., :-1, :].contiguous()
# print(f"shift_labels.shape: {shift_labels.shape}")
# print(f"shift_logits.shape: {shift_logits.shape}")
# loss_fct = CrossEntropyLoss(reduction='none')
# loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
# print(f"loss.shape: {loss.shape}")
# loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
# print(f"loss_per_sample.shape: {loss_per_sample.shape}")
# weight_mask = torch.stack([(inputs == kt).float() for kt in keytoken_ids])
# print(f"weight_mask.shape: {weight_mask.shape}")
# weights = weight_mask.sum(axis=[0, 2])
# weights = alpha * (1.0 + weights)
# weighted_loss = (loss_per_sample * weights).mean()
# print(f"weighted_loss: {weighted_loss}")

In [ ]:
%%writefile main.py

import math
import torch
from huggingface_hub import HfApi, create_repo
from transformers import AutoTokenizer
from transformers import get_scheduler
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.distributed import DistributedSampler
from accelerate import Accelerator
from accelerate.utils import set_seed as accl_set_seed
from src.config import *
from src.utils import hf_login, get_mp
from src.data import load_raw_datasets, create_tokenized_datasets
from src.train import load_model, get_grouped_params
from src.evaluation import keytoken_weighted_loss, get_keytoken_ids, run_eval


if __name__ == "__main__":
    # Init accelerator
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp(),
    )
    accl_set_seed(42)
    
    if accelerator.is_main_process:
        accelerator.print(f"checkpoint name: {ckpt_name}")
        accelerator.print(f"# GPUs: {accelerator.num_processes}")
        effective_bs = train_bs * gacc_steps * accelerator.num_processes
        accelerator.print(f"effective_batch_size: {effective_bs}")
    # HF Login and repo initiation
    with accelerator.main_process_first():
        hf_login()
        
    if accelerator.is_main_process:
        hf_api = HfApi()
        accelerator.print(f"HF user: {hf_api.whoami()['name']}")
        if push_to_hub:
            # Create the HF repo to push the model to the HF hub
            HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
            create_repo(repo_id=HF_REPO_ID, exist_ok=True)
            accelerator.print(f"HF repo: {HF_REPO_ID}")

    # Load model, tokenizer, and create dataloaders
    with accelerator.main_process_first():
        raw_datasets = load_raw_datasets()
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)
        tokenizer.pad_token = tokenizer.eos_token
        tokenized_datasets = create_tokenized_datasets(tokenizer, raw_datasets)
        model = load_model(tokenizer)
        tokenized_datasets.set_format("torch")
        train_dataloader = DataLoader(
            tokenized_datasets["train"], batch_size=train_bs,
            shuffle=True, drop_last=True,
        )
        eval_dataloader = DataLoader(
            tokenized_datasets["valid"], batch_size=eval_bs,
            shuffle=False
        )

    keytoken_ids = get_keytoken_ids(tokenizer)

    if accelerator.is_main_process:
        train_ds = train_dataloader.dataset
        eval_ds = eval_dataloader.dataset
        model_size = sum(t.numel() for t in model.parameters())
        accelerator.print(f"GPT-2 size: {model_size/1e6:.1f}M parameters")
        accelerator.print(f"final input datasets features: {train_ds.features}")
        accelerator.print(f"train DS length: {len(train_ds)}")
        accelerator.print(f"eval DS length: {len(eval_ds)}")
        print(f"keytoken_ids: {keytoken_ids}")

    # Preparing training components
    optimizer = AdamW(get_grouped_params(model), lr=lr)
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = max(1, num_update_steps_per_epoch // 4)
    eval_steps = num_update_steps_per_epoch
    
    
    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=logging_steps,# only because they are equal here
        num_training_steps=num_training_steps,
    )

    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    pid = f"[rank={rank} main={is_main}]"
    # accelerator.print() only prints from rank 0. Use print directly so both processes print:
    print(f"{pid}|train_len={len(train_dataloader)}|eval_len={len(eval_dataloader)}")
    print(f"{pid}|eval_steps={eval_steps}|train_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone()

    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
        leave=False,
    )

    if eval_on_start:
        eval_logs = run_eval(model, eval_dataloader, accelerator)
        accelerator.print(f"Step {global_step} eval:", eval_logs)

    
    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    logits = model(batch["input_ids"]).logits
                    loss = keytoken_weighted_loss(
                        batch["input_ids"], logits, keytoken_ids
                    )
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                do_sync = accelerator.sync_gradients
                if do_sync:
                    accelerator.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if do_sync:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    loss_all = accelerator.gather_for_metrics(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather_for_metrics(lr_t).detach().cpu().tolist()
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            msg = f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                            accelerator.print(msg)
                    running_loss = 0.0
                    running_mb = 0
                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(model, eval_dataloader, accelerator)
                    msg = f"Step {global_step} eval:", eval_logs
                    if accelerator.is_main_process: accelerator.print(msg)

    if accelerator.is_main_process:
        accelerator.print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training()
    accelerator.free_memory()

In [ ]:
! accelerate launch --num_processes 2 main.py